1. **What's the difference between inner, left, and full joins?**

    - **INNER JOIN**--> Returns rows where the join condition matches on both sides.
    - **FULL JOIN**--> Returns all rows from both DataFrames. Unmatched columns from either side contain NULL.
    - **LEFT JOIN**--> Returns all rows from the left DataFrame and matching rows from the right. Unmatched right-side columns contain NULL.

        eg- If one customer has three accounts, that customer can produce three output rows.

2. **What's the difference between left_semi and inner?**

    - **Inner** --> fetches MATCHING ROWS OF BOTH left and Right.
    - **Left_Semi** --> fetches ONLY MATCHING ROWS from LEFT TABLE. DOES NOT INCLUDE right table rows/columns. eg- 1001, 1002, 1003 (1004/5/6 excluded)
            A left_semi join returns rows from the left DataFrame for which a match exists in the right DataFrame, but it returns only columns from the left DataFrame.

3. **What does left_anti do?**

    **Left_Anti** --> similar to SQL - NOT EXISTS cmd. fetches all the rows from left table (customers) that do not exists in Right table(Accounts)
    eg- 1004, 1005

4. **Why can a join increase the number of records?**

    A join can increase the row count when the join key isn't unique on one or both sides. For example, if one customer has multiple accounts, joining Customer to Account produces one output row for each matching account. A many-to-many relationship can increase the row count even more.
    EG- Customer --> 1001 --> John
        Accounts --> 1001 Account A
                     1001 Account B
        **After Joining**--> 1001 John Account A
                          1001 John Account B

5. **How do you handle duplicate column names after a Spark join?**
    
    I alias the DataFrames so duplicate column names can be referenced unambiguously, then explicitly select() the columns needed in the output and rename them with alias() when necessary.

6. **What's the difference between groupBy() and agg()?**
    groupBy() --> itself doesn't perform the aggregation.
                  It defines how the records should be grouped.. aggregates the data and returns ONE ROW PER GROUP
    agg() --> Spark what calculations to perform on each group.

    groupBy() defines the **grouping key or keys**, while agg() **specifies the aggregation** functions such as count, sum, min, max, or average that Spark should calculate for each group.

    ````
    eg-
        Aggregator transformation

        Group By port
            ↓
        custId

        Aggregate expressions
            ↓
        COUNT()
        SUM()
        AVG()
    ````

7. **Why can Spark joins be expensive?**
Spark joins can be expensive because matching records may reside on different partitions/executors. Spark may need to redistribute data based on the join key so matching records end up together. This causes network I/O, disk I/O, serialization and additional computation.

8. **- What is a shuffle?**

    Redistributing data across Spark partitions/executors.

    A shuffle is the **redistribution** of data across Spark partitions, often across executors, so_ related records can be processed together_. Operations such as **joins, groupBy, distinct and repartition** may trigger shuffles. Shuffles are expensive because they involve network transfer and may also involve serialization and disk I/O.

9. ### Senior-level scenario

**You have a 500-million-row transaction DataFrame and a 50,000-row product lookup table. How would you approach the join efficiently?**

Since the transaction DataFrame contains 500 million rows while the product lookup contains only 50,000 rows, 
- I would first evaluate the actual **size of the lookup DataFrame**. If it's small enough to broadcast safely, I would use a** broadcast join** so Spark sends a copy of the product lookup to the executors processing the transaction partitions. This can avoid shuffling the large transaction DataFrame. 
- I would also select only the required lookup columns before broadcasting and check the execution plan to verify the intended join strategy.